# Module 12 — Model Development and Evaluation

**Assumption:** learners already know supervised learning, train/validation/test splits, standard metrics, cross-validation, feature engineering, and model training.

This notebook focuses on **evaluation as an engineering and decision system - evaluation design, statistical uncertainty, business decision-making, robustness, and deployment-readiness evidence.**, not on teaching basic metrics.

## Coverage
1. Evaluation design and dataset contracts
2. IID(Independent and Identically Distributed) vs temporal vs grouped validation
3. Leakage detection
4. Repeated / nested evaluation
5. Bootstrap confidence intervals and paired model comparison
6. Threshold optimization and business-cost evaluation
7. Probability calibration
8. Ranking evaluation
9. Regression evaluation under asymmetric business loss
10. Subgroup / slice evaluation and fairness diagnostics
11. Robustness and perturbation testing
12. Benchmark harness for multiple models
13. Error analysis and error taxonomy
14. Out-of-distribution / stress evaluation
15. Reproducible evaluation reports
16. A capstone evaluation pipeline

The examples deliberately use a synthetic dataset so the notebook is self-contained.


In [2]:
# Core imports
import numpy as np
import pandas as pd

from dataclasses import dataclass
from typing import Callable, Dict, Iterable, List, Optional, Tuple

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupKFold, TimeSeriesSplit,
    cross_val_predict, cross_validate
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, log_loss,
    brier_score_loss, confusion_matrix, classification_report,
    precision_recall_curve, roc_curve, mean_absolute_error,
    mean_squared_error, r2_score
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

## 1. Build an evaluation dataset with realistic failure modes
Evaluation datasets should reflect:
- population drift,
- class imbalance,
- entity/group dependence,
- time dependence,
- missingness,
- noisy labels,
- rare but costly cases.

The synthetic data below contains **customers observed repeatedly over time**, which lets us demonstrate why random row-wise splitting can be misleading.

The notebook creates data that includes properties often found in business datasets:
- Repeated observations from the same customer.
- Relationships that change over time.
- Missing feature values.
- A binary outcome.
- Customer-level factors that influence the outcome.
This lets us examine whether a validation strategy matches the way a model will actually be used.

The important distinction is between the observed features and the data-generating process. The model does not observe the latent customer effect directly, but that effect still influences the target.
This is useful for discussing repeated measures and the difference between evaluating on known entities and unseen entities.

Limitation: Although the dataset contains a time variable, its feature values are generated independently of time. The example is useful for demonstrating validation mechanics, but it is not a realistic simulation of every form of temporal drift.


In [3]:
# Synthetic longitudinal customer dataset
n_customers = 6000
observations_per_customer = 4
n = n_customers * observations_per_customer #This produces 24,000 rows: four observations for each of 6,000 customers.

#customer_id repeats each customer ID four times.
#period assigns the values 0, 1, 2, and 3 to each customer's observations.

customer_id = np.repeat(np.arange(n_customers), observations_per_customer)
period = np.tile(np.arange(observations_per_customer), n_customers)

x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
x3 = rng.normal(size=n)
x4 = rng.normal(size=n)

# Customer-specific latent effect -> creates within-customer dependence. Each customer gets a hidden effect that is shared across their four observations. This creates dependence between observations from the same customer.
#Customer A may naturally have higher risk than average. That hidden “higher-risk tendency” is added to all 4 observations of Customer A.
customer_effect = rng.normal(0, 0.9, size=n_customers)
latent = customer_effect[customer_id]

#The logit combines feature effects, customer effect, and time effect. The sigmoid transformation converts it into a probability between 0 and 1. rng.binomial then samples a binary target using that probability.
logit = ( 0.9 * x1 - 0.7 * x2 + 0.4 * x3 + 0.2 * x4 + latent + 0.25 * period )

p = 1 / (1 + np.exp(-logit))
y = rng.binomial(1, p)              #customer_effect is used indirectly when creating target.

df = pd.DataFrame({
    "customer_id": customer_id,
    "period": period,
    "x1": x1,
    "x2": x2,
    "x3": x3,
    "x4": x4,
    "target": y    })

# Introduce missing values
for c in ["x2", "x4"]:
    mask = rng.random(n) < 0.04
    df.loc[mask, c] = np.nan

df.head(), df["target"].mean()


(   customer_id  period        x1        x2        x3        x4  target
 0            0       0  0.304717 -0.422500 -0.208057 -0.966290       1
 1            0       1 -1.039984  0.969549  0.106168 -0.900647       0
 2            0       2  0.750451 -0.432650 -0.245893  0.512680       1
 3            0       3  0.940565  0.354970 -0.303504       NaN       1
 4            1       0 -1.951035 -0.125404 -0.833135  1.735438       0,
 np.float64(0.5640833333333334))

## 2. Dataset contract and evaluation invariants

An evaluation dataset should have explicit invariants rather than being treated as an arbitrary DataFrame.

Typical contract:

- target definition and observation window are fixed;
- features are timestamp-valid;
- entity IDs are available where leakage is possible;
- labels are frozen for the evaluation run;
- test data is never used to tune thresholds or hyperparameters;
- transformations are fitted only on the training partition;
- metrics are computed from raw predictions, not manually edited outputs.

*An evaluation dataset should have a documented contract. Otherwise, two teams may evaluate the same model on different populations, target definitions, time windows, or feature versions and report incomparable results.*

This is a simple example of converting evaluation assumptions into executable checks rather than relying entirely on documentation.
In an enterprise implementation, the contract could also verify:
- Label definition and prediction horizon.
- Feature availability timestamps.
- Duplicate entity/time keys.
- Permitted value ranges and missingness limits.
- Dataset and feature versions.
- Whether labels are mature enough to evaluate.

Important limitation: The current validator checks column existence and allowed prediction-type names. It does not verify most of the invariants listed in the Markdown cell. It is a starting point, not a complete production validator.

In [4]:
@dataclass
class EvaluationContract:
    target: str
    id_columns: List[str]
    time_column: Optional[str]
    prediction_type: str  # "probability", "score", "label"
    positive_label: int = 1

    def validate(self, data: pd.DataFrame):
        assert self.target in data.columns, f"Missing target: {self.target}"
        for c in self.id_columns:
            assert c in data.columns, f"Missing ID column: {c}"
        if self.time_column:
            assert self.time_column in data.columns
        assert self.prediction_type in {"probability", "score", "label"}
        return True

contract = EvaluationContract(
    target="target",
    id_columns=["customer_id"],
    time_column="period",
    prediction_type="probability"
)

contract.validate(df)
print("Evaluation contract valid.")



Evaluation contract valid.


## 3. Leakage: demonstrates how random row splitting can inflate performance

If the same customer appears in both train and validation, the model can exploit customer-specific structure.

This is not necessarily a bug in the algorithm. It is a **validation design bug** if production predicts on unseen customers.

A random split can be inappropriate when multiple rows belong to the same customer, patient, device, store, or other entity.

If the deployment objective is to predict outcomes for unseen customers, the evaluation should generally keep all observations for each customer in one partition.

The two experiments represent different deployment questions:
- Random row split: How well does the model perform when future records may belong to customers also represented in training?
- Group split: How well does the model generalize to customers whose records were not used for fitting?

In [5]:
features = ["x1", "x2", "x3", "x4"]

#The pipeline performs median imputation, standardization, and logistic regression.
#Keeping preprocessing inside the pipeline means that the imputer and scaler are fitted on the training data, rather than on the full dataset.

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

# Naive random row split The first experiment uses a stratified random row split: stratify attempts to preserve the target-class proportion across the two partitions.
X_train, X_test, y_train, y_test = train_test_split(
    df[features], df["target"],
    test_size=0.25,
    stratify=df["target"],
    random_state=RANDOM_STATE
)

pipe.fit(X_train, y_train)
p_random = pipe.predict_proba(X_test)[:, 1]

print("Random row split ROC-AUC:", roc_auc_score(y_test, p_random))
print("Random row split PR-AUC :", average_precision_score(y_test, p_random))

# Group split: no customer crosses train/test
groups = df["customer_id"].values
gkf = GroupKFold(n_splits=5)

group_scores = []
for train_idx, test_idx in gkf.split(df[features], df["target"], groups=groups):
    pipe.fit(df.iloc[train_idx][features], df.iloc[train_idx]["target"])
    p_hat = pipe.predict_proba(df.iloc[test_idx][features])[:, 1]
    group_scores.append(roc_auc_score(df.iloc[test_idx]["target"], p_hat))

print("Group CV ROC-AUC:", np.mean(group_scores), "+/-", np.std(group_scores))

Random row split ROC-AUC: 0.7548155648816883
Random row split PR-AUC : 0.7896969609203831
Group CV ROC-AUC: 0.7500095989960103 +/- 0.0084182636269335


## 4. Temporal validation

For systems where the model is trained on the past and deployed into the future, a random split is often the wrong abstraction.

The evaluation question becomes:

> If I train using information available at time T, how well do I perform on T+1?

This section demonstrates expanding-window evaluation.

-Compare fold-by-fold results rather than relying only on the average. A declining metric may indicate temporal instability, a changing population, or sampling variation.

Important limitation: The synthetic dataset has only four periods, but TimeSeriesSplit divides the ordered rows into folds. It does not explicitly enforce whole-period boundaries. A training fold can therefore contain some customers' period-1 records while a test fold contains other customers' period-1 records. For rigorous business evaluation, split by actual timestamps or time windows so the test period is strictly later than the training period

In [6]:
# Sort by period to make temporal ordering explicit. The dataset is sorted by period, and TimeSeriesSplit creates four expanding-window folds. For each fold, the model is trained on earlier rows and evaluated on later rows.
temporal_df = df.sort_values(["period", "customer_id"]).reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=4)
temporal_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(temporal_df), start=1):
    Xtr = temporal_df.iloc[train_idx][features]
    ytr = temporal_df.iloc[train_idx]["target"]
    Xte = temporal_df.iloc[test_idx][features]
    yte = temporal_df.iloc[test_idx]["target"]

    pipe.fit(Xtr, ytr)
    p_hat = pipe.predict_proba(Xte)[:, 1]

    temporal_scores.append({
        "fold": fold,
        "train_end": temporal_df.iloc[train_idx]["period"].max(),
        "test_period_min": temporal_df.iloc[test_idx]["period"].min(),
        "test_period_max": temporal_df.iloc[test_idx]["period"].max(),
        "roc_auc": roc_auc_score(yte, p_hat),
        "pr_auc": average_precision_score(yte, p_hat),
        "log_loss": log_loss(yte, p_hat)
    })

pd.DataFrame(temporal_scores)


,fold,train_end,test_period_min,test_period_max,roc_auc,pr_auc,log_loss
0,1,0,0,1,0.749438,0.758214,0.592766
1,2,1,1,2,0.752373,0.794805,0.588140
2,3,2,2,3,0.755311,0.821997,0.580209
3,4,3,3,3,0.751820,0.826385,0.579529


## 5. Repeated evaluation and confidence intervals

A single metric such as `ROC-AUC = 0.84` is incomplete.  A metric such as ROC-AUC = 0.84 is only a point estimate. Its reliability depends on the evaluation sample and the population being evaluated.

A confidence interval gives an indication of how much the estimate varies under a specified resampling procedure.

For decision making, report:

- central estimate,
- uncertainty,
- evaluation population,
- split strategy,
- sample size,
- positive prevalence,
- metric definition.

Bootstrap confidence intervals are useful when the metric distribution is not conveniently Gaussian.

The interval helps answer whether the observed performance estimate is precise enough for the decision at hand. It does not tell us that the model will perform within the interval in every future deployment.

Important limitation: This is a row-level bootstrap. Because the data has repeated customer observations, rows are not necessarily independent. If the evaluation unit is a customer, a cluster bootstrap that resamples customers and keeps their observations together may be more appropriate.
#Also, the interval captures resampling uncertainty for the fixed fitted model and evaluation set. It does not automatically include uncertainty from retraining, model selection, or future distribution shifts.

In [ ]:
def bootstrap_ci(  y_true,   y_score,
    metric_fn: Callable,
    n_bootstrap: int = 3000,
    confidence: float = 0.95,
    random_state: int = 42 ):                           #The function bootstrap_ci() repeatedly samples test-set rows with replacement:
    rng_local = np.random.default_rng(random_state)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    values = []
    n_obs = len(y_true)

    for _ in range(n_bootstrap):
        idx = rng_local.integers(0, n_obs, n_obs)
        yt = y_true[idx]
        ys = y_score[idx]

        # Some bootstrap samples may contain only one class.
        if len(np.unique(yt)) < 2:
            continue

        values.append(metric_fn(yt, ys))

    alpha = 1 - confidence
    return {
        "estimate": metric_fn(y_true, y_score),  #estimate: the metric on the original evaluation sample.
        "lower": np.quantile(values, alpha / 2),  #lower: the lower percentile bound.
        "upper": np.quantile(values, 1 - alpha / 2), #pper: the upper percentile bound.
        "n_bootstrap": len(values)                      #n_bootstrap: the number of valid resamples.
    }

ci_auc = bootstrap_ci(y_test, p_random, roc_auc_score)
ci_ap = bootstrap_ci(y_test, p_random, average_precision_score)

pd.DataFrame([ci_auc, ci_ap], index=["ROC-AUC", "PR-AUC"])

#ROC-AUC measures how well the model separates the two classes, while PR-AUC focuses on identifying positive cases with good precision.

,estimate,lower,upper,n_bootstrap
ROC-AUC,0.754816,0.742958,0.766766,3000
PR-AUC,0.789697,0.775017,0.803765,3000


## 6. Paired bootstrap comparison of two models

When comparing models, use the **same evaluation examples**.

Comparing:

- Model A: AUC = 0.82
- Model B: AUC = 0.84

does not tell us whether the improvement is practically or statistically meaningful.

A paired bootstrap asks whether the difference remains positive under repeated resampling of the same observations.

In [ ]:
model_a = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

model_b = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

model_a.fit(X_train, y_train)
model_b.fit(X_train, y_train)

pa = model_a.predict_proba(X_test)[:, 1]
pb = model_b.predict_proba(X_test)[:, 1]

#A bootstrap sample is a new dataset created by randomly selecting rows from the original dataset with replacement.
#paired_bootstrap_difference repeats this 3,000 times and returns the full-sample metric values, the observed difference, a 95% percentile interval for the difference, and the fraction of bootstrap differences greater than zero.

def paired_bootstrap_difference(y_true, score_a, score_b, metric_fn, n_bootstrap=3000):
    rng_local = np.random.default_rng(123)
    y_true = np.asarray(y_true)
    score_a = np.asarray(score_a)
    score_b = np.asarray(score_b)

    diffs = []
    for _ in range(n_bootstrap):
        idx = rng_local.integers(0, len(y_true), len(y_true))
        yt = y_true[idx]
        if len(np.unique(yt)) < 2:
            continue
        diffs.append(
            metric_fn(yt, score_b[idx]) - metric_fn(yt, score_a[idx])
        )

    diffs = np.asarray(diffs)
    return {
        "model_A": metric_fn(y_true, score_a),
        "model_B": metric_fn(y_true, score_b),
        "delta_B_minus_A": metric_fn(y_true, score_b) - metric_fn(y_true, score_a),  #B's ROC-AUC − A's ROC-AUC; positive means B scores higher
        "ci_lower": np.quantile(diffs, 0.025),
        "ci_upper": np.quantile(diffs, 0.975),
        "prob_B_better": np.mean(diffs > 0)     #Interval entirely above 0 → evidence favors B. Interval entirely below 0 → evidence favors A. Interval includes 0 → difference is less conclusive.
    }

paired_bootstrap_difference(y_test, pa, pb, roc_auc_score)

#Pairing matters because both models are evaluated on the same difficult and easy examples. That reduces noise in the comparison relative to comparing metrics calculated on unrelated test sets.
#Interpret the interval around the difference:
#- If it lies entirely above zero, the results support model B having a higher metric under this resampling analysis.
#- If it lies entirely below zero, the results support model A.
#- If it includes zero, the evidence for a difference is less conclusive.
#This is not a substitute for assessing business value. A small AUC improvement may not justify greater inference cost or latency.
#Technical caveat: The function uses prob_B_better as the fraction of bootstrap differences above zero. This is a useful descriptive quantity, but it should not be interpreted automatically as a formal Bayesian probability that B is superior.

{'model_A': 0.7548155648816883,
 'model_B': 0.7434454999602373,
 'delta_B_minus_A': -0.011370064921450984,
 'ci_lower': np.float64(-0.016109037014372194),
 'ci_upper': np.float64(-0.006722376533999236),
 'prob_B_better': np.float64(0.0)}

## 7. Threshold selection is a business decision

A probability model and a decision policy are different artifacts.

For example:

- Model: predicts probability of default.
- Policy: approve / review / reject.

The threshold should be selected using:

- false-positive cost,
- false-negative cost,
- capacity constraints,
- intervention cost,
- regulatory constraints,
- downstream operational capacity.

Do not select a threshold on the final test set.

In [9]:
#expected_cost() converts probabilities into binary predictions at a chosen threshold: (0.01, 0.99, 99)
def expected_cost(y_true, y_prob, threshold, fp_cost=1.0, fn_cost=5.0):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    total_cost = fp * fp_cost + fn * fn_cost
    return {
        "threshold": threshold,
        "cost": total_cost,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "tn": tn,
        "positive_rate": y_pred.mean()
    }

threshold_grid = np.linspace(0.01, 0.99, 99)

#The total cost is:  FP * {cost of FP} + FN * {Cost of FN}. The notebook tests thresholds from 0.01 through 0.99 and uses false-positive cost 2 and false-negative cost 10.
costs = pd.DataFrame([
    expected_cost(y_test, p_random, t, fp_cost=2, fn_cost=10)
    for t in threshold_grid
])

best_policy = costs.loc[costs["cost"].idxmin()]
best_policy

threshold           0.140000
cost             5122.000000
fp               2511.000000
fn                 10.000000
tp               3374.000000
tn                105.000000
positive_rate       0.980833
Name: 13, dtype: float64

## 8. Capacity-constrained decisioning

Sometimes the business can only process 5% of cases.

Then threshold selection becomes a ranking problem:

> Which 5% should receive the intervention?

This is common in:

- fraud review,
- retention campaigns,
- collections,
- sales prioritization,
- manual risk review.

In [ ]:
#top_k_policy_metrics() sorts scores from highest to lowest:

def top_k_policy_metrics(y_true, y_score, fractions=(0.01, 0.05, 0.10, 0.20)):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    order = np.argsort(-y_score)
    y_sorted = y_true[order]

    rows = []
    total_positives = y_true.sum()

    #For each requested fraction, it selects the top-scoring cases and computes:
    #- precision_at_k: proportion of selected cases that are positive.
    #- recall_at_k: proportion of all positive cases captured by the selected group.
    #- lift: precision in the selected group divided by the overall positive rate.

    for f in fractions:
        k = max(1, int(len(y_true) * f))
        captured = y_sorted[:k].sum()
        precision_at_k = captured / k
        recall_at_k = captured / total_positives if total_positives else np.nan

        rows.append({
            "top_fraction Reviewed": f,
            "Transactions reviewed(k)": k,
            "precision_at_k": precision_at_k,
            "recall_at_k": recall_at_k,
            "lift": precision_at_k / y_true.mean()
        })

    return pd.DataFrame(rows)

top_k_policy_metrics(y_test, p_random)

#Suppose the review team can inspect 5% of transactions. Precision at 5% answers, “What fraction of the cases we review are actually positive?” Recall at 5% answers, “What fraction of all positive cases did we manage to capture?”
#Lift indicates how much more concentrated the selected group is in positive cases compared with random selection.
#Technical caveat: The code uses the true target labels to calculate retrospective performance, which is appropriate for offline evaluation. At inference time, only the scores are available. Also, lift is undefined in principle when the overall positive rate is zero; the current function does not guard against that case.

#Output Interpretation
#Precision decreases: Reviewing more transactions means including more lower-ranked cases, so the proportion of positives falls.
#Recall increases: Reviewing more transactions captures a larger share of all positive cases.
#Lift decreases: The selected group becomes less concentrated in positives compared with random selection. Even at 20%, it has about 1.52 times the overall positive rate.
#Important: High precision does not mean high overall coverage. The team finds only a small fraction of all positives at the 5% capacity level.

,top_fraction Reviewed,Transactions reviewed(k),precision_at_k,recall_at_k,lift
0,0.01,60,0.966667,0.017139,1.713948
1,0.05,300,0.920000,0.081560,1.631206
2,0.10,600,0.895000,0.158688,1.586879
3,0.20,1200,0.856667,0.303783,1.518913


## 9. Calibration: ranking quality is not probability quality

A model can have excellent ROC-AUC while producing poor probabilities.

If a model says:

> "Among cases scored around 0.8, roughly 80% should actually be positive."

that is a calibration statement.

Calibration matters when probabilities feed:

- expected-loss calculations,
- capital allocation,
- pricing,
- risk limits,
- resource planning,
- downstream models.

In [ ]:
#The function calibration_table() divides predicted probabilities into ten equal-width bins from 0 to 1.
#For each occupied bin, it calculates:
#	Mean predicted probability.
#	Observed positive rate.
#	Number of observations.
#	Absolute gap between predicted probability and observed rate.
#	A large gap indicates a discrepancy between the average predicted risk and the observed outcome frequency for that bin.


def calibration_table(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    bucket = pd.cut(y_prob, bins=bins, include_lowest=True)

    tmp = pd.DataFrame({"y": y_true, "p": y_prob, "bin": bucket})
    out = tmp.groupby("bin", observed=True).agg(
        predicted_probability=("p", "mean"),
        observed_rate=("y", "mean"),
        count=("y", "size")
    ).reset_index()

    out["absolute_gap"] = (
        out["predicted_probability"] - out["observed_rate"]
    ).abs()

    return out

calibration_table(y_test, p_random)

#Output Interpretation
#This table checks whether predicted probabilities match actual outcome frequencies — in other words, whether the model is well calibrated.
#Key observations
#- Good calibration overall: Most bins have small gaps between predicted probability and observed rate.
#- Largest mismatch: In the 0.2–0.3 bin, predicted probability is 25.34%, but the observed positive rate is 20.75%. The model overestimates risk by about 4.59 percentage points.
#- Slight underestimation: In the 0.5–0.6 bin, the observed rate (57.71%) is higher than the predicted probability (55.11%).
#- Strong calibration at high probabilities: The 0.8–0.9 and 0.9–1.0 bins have very small gaps.
#- Sample sizes vary: The bins contain different numbers of observations, so gaps in smaller bins may be less reliable.
#Example interpretation
#For the 0.6–0.7 bin, the model predicts an average probability of 65.06%, and approximately 65.41% of cases are actually positive. This is good calibration.
#Bottom line: The model appears reasonably well calibrated, with its largest discrepancy in the 0.2–0.3 probability range. Calibration is about the accuracy of predicted probabilities, not how accurately the model classifies individual cases.

,bin,predicted_probability,observed_rate,count,absolute_gap
0,"(-0.001, 0.1]",0.078910,0.081081,37,0.002172
1,"(0.1, 0.2]",0.156362,0.164794,267,0.008432
2,"(0.2, 0.3]",0.253426,0.207506,453,0.045921
3,"(0.3, 0.4]",0.350990,0.343662,710,0.007328
4,"(0.4, 0.5]",0.451323,0.435162,802,0.016161
5,"(0.5, 0.6]",0.551130,0.577131,927,0.026001
6,"(0.6, 0.7]",0.650562,0.654082,980,0.003519
7,"(0.7, 0.8]",0.749644,0.745685,869,0.003960
8,"(0.8, 0.9]",0.844235,0.844567,727,0.000331
9,"(0.9, 1.0]",0.930704,0.929825,228,0.000880


In [ ]:
# Compare raw logistic probabilities with isotonic calibration.Isotonic calibration learns a flexible mapping from model scores to estimated probabilities. The calibrated model is then evaluated using Brier score and log loss.
#	Brier score: mean squared error between predicted probabilities and binary outcomes. Actual outcome = 1 (positive case), Predicted probability = 0.8, Squared error = ((0.8-1)^2 = 0.04).Result is: 0.20284 → 0.20278 after calibration. A tiny improvement.
#	Log loss: penalizes inaccurate probabilities, especially confident predictions that are wrong. Ex- Actual outcome = 1,Predict 0.9 → low penalty, Predict 0.1 → high penalty, Predict 0.001 → extremely high penalty, Result: 0.59255 → 0.59694 after calibration. It became slightly worse.
# 	Lower is generally better for both metrics.

calibrated_rf = CalibratedClassifierCV(
    estimator=RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    method="isotonic",
    cv=5
)

calibrated_rf.fit(X_train, y_train)
p_cal = calibrated_rf.predict_proba(X_test)[:, 1]

print("Uncalibrated RF Brier:", brier_score_loss(y_test, pb))
print("Calibrated RF Brier  :", brier_score_loss(y_test, p_cal))
print("Uncalibrated RF LogLoss:", log_loss(y_test, pb))
print("Calibrated RF LogLoss  :", log_loss(y_test, p_cal))

#Technical caveat: The comparison uses Brier score and log loss on the held-out test set, which is appropriate as a final comparison if the test set was not used to choose calibration settings. Calibration does not guarantee an improvement on every dataset. The calibration model's cross-validation must also match the dependence structure of the data; ordinary folds can be problematic for grouped or temporal observations.

Uncalibrated RF Brier: 0.202837671731761
Calibrated RF Brier  : 0.20277756286295953
Uncalibrated RF LogLoss: 0.5925511534652961
Calibrated RF LogLoss  : 0.5969373371349257


## 10. A unified classification evaluation function

This is the type of function that can become part of an organization's standard model-evaluation library.

It deliberately reports **more than one metric** and keeps threshold-dependent and threshold-independent metrics separate.

In [13]:
#This function deliberately separates two metric families:
#Threshold-independent metrics: ROC-AUC, average precision, log loss, and Brier score use the probability scores.
#Threshold-dependent metrics: Precision, recall, specificity, accuracy, and F1 depend on the selected decision threshold.
#This distinction is useful when comparing models independently of a particular policy and then evaluating the policy itself.
#Technical caveat: The function accepts positive_label, but the current implementation does not use that argument; it assumes binary labels 0 and 1. It also assumes probabilities are valid and that both classes are present. A production library should validate these conditions and handle degenerate cases explicitly.

def evaluate_binary_classifier(
    y_true,
    y_prob,
    threshold=0.5,
    positive_label=1
):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "n": len(y_true),
        "prevalence": y_true.mean(),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "log_loss": log_loss(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "threshold": threshold
    }

evaluate_binary_classifier(y_test, p_random, threshold=0.5)



{'n': 6000,
 'prevalence': np.float64(0.564),
 'roc_auc': 0.7548155648816883,
 'pr_auc': 0.7896969609203831,
 'log_loss': 0.5804866084878836,
 'brier': 0.19834414874342082,
 'accuracy': 0.6975,
 'balanced_accuracy': 0.6849353135098792,
 'precision': 0.7102653444116859,
 'recall': 0.783096926713948,
 'specificity': np.float64(0.5867737003058104),
 'f1': 0.7449051300070274,
 'tn': np.int64(1535),
 'fp': np.int64(1081),
 'fn': np.int64(734),
 'tp': np.int64(2650),
 'threshold': 0.5}

## 11. Slice / subgroup evaluation

Global metrics can hide catastrophic behavior for a subgroup.

Slice dimensions might include:

- geography,
- customer segment,
- device,
- channel,
- product,
- acquisition source,
- age band,
- tenure,
- language,
- transaction size.

The important engineering pattern is to make slice evaluation **automatic**, not an ad-hoc manual exercise.

In [ ]:
# Create illustrative slices
eval_df = df.loc[X_test.index].copy()
eval_df["prediction_probability"] = p_random
eval_df["prediction"] = (p_random >= 0.5).astype(int)

eval_df["segment"] = np.where(eval_df["x1"].fillna(0) > 0, "A", "B")
eval_df["risk_band"] = pd.qcut(
    eval_df["x2"].fillna(eval_df["x2"].median()),
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

def slice_metrics(data, slice_col):
    rows = []
    for value, g in data.groupby(slice_col, observed=True):
        if g["target"].nunique() < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(g["target"], g["prediction_probability"])

        rows.append({
            "slice": slice_col,
            "value": value,
            "n": len(g),
            "prevalence": g["target"].mean(),       #Percentage of actual positive cases in the group.
            "roc_auc": auc,                         #How well the model ranks positive cases above negative cases.
            "precision": precision_score(           #Of predicted positives, how many are truly positive?
                g["target"], g["prediction"], zero_division=0
            ),
            "recall": recall_score(                 #Of all actual positives, how many did the model identify?
                g["target"], g["prediction"], zero_division=0
            ),
            "mean_score": g["prediction_probability"].mean()        #Average predicted probability across the group.
        })

    return pd.DataFrame(rows)

pd.concat([
    slice_metrics(eval_df, "segment"),
    slice_metrics(eval_df, "risk_band")
], ignore_index=True)

#This is the foundation of automated slice monitoring. In a real system, slices could represent geography, channel, product, customer tenure, device, or another business-relevant dimension.
#Check whether a subgroup's poor performance is caused by sample size, a different outcome prevalence, different feature quality, model behavior, or the decision threshold.
#Important caveat: These slices are synthetic and not automatically meaningful business or fairness groups. A slice difference is a diagnostic signal, not proof of unfairness or a causal explanation. Small groups need uncertainty estimates, and any real-world fairness assessment requires careful choice of appropriate metrics and context.


,slice,value,n,prevalence,roc_auc,precision,recall,mean_score
0,segment,A,3028,0.695178,0.710655,0.744584,0.930641,0.690987
1,segment,B,2972,0.430350,0.715419,0.628182,0.540266,0.438820
2,risk_band,Q1,1500,0.715333,0.742614,0.761544,0.937558,0.719228
3,risk_band,Q2,1618,0.592089,0.726082,0.696707,0.839248,0.602834
4,risk_band,Q3,1382,0.529667,0.710912,0.667097,0.706284,0.528152
5,risk_band,Q4,1500,0.414000,0.733242,0.671518,0.520129,0.408234


## 12. Error taxonomy

Counting errors is not enough.

A useful error-analysis loop asks:

1. What type of error occurred?
2. Is the error systematic?
3. Is it caused by data, label, feature, model, threshold, or workflow?
4. Is the error recoverable?
5. What intervention would reduce it?
6. What is the cost of fixing it?

The next cell creates a structured error-analysis table.

In [15]:
errors = eval_df.copy()
errors["error_type"] = np.select(
    [
        (errors["target"] == 1) & (errors["prediction"] == 0),
        (errors["target"] == 0) & (errors["prediction"] == 1),
        (errors["target"] == 1) & (errors["prediction"] == 1),
        (errors["target"] == 0) & (errors["prediction"] == 0)
    ],
    ["false_negative", "false_positive", "true_positive", "true_negative"],
    default="unknown"
)

errors["confidence"] = np.abs(errors["prediction_probability"] - 0.5) #⦁	It calculates confidence as prediction_probability - 0.5)
#This is distance from the 0.5 decision threshold. It is a simple measure of how far a probability lies from the classification boundary, not a universal measure of model certainty.


errors.groupby("error_type").agg(
    count=("target", "size"),
    mean_confidence=("confidence", "mean"),
    mean_score=("prediction_probability", "mean")
).sort_values("count", ascending=False)

#A false negative and a false positive can have very different business consequences. The right analysis connects error type to cost, customer impact, root cause, and potential remedy.
#Technical caveat: The confidence calculation is only meaningful relative to the chosen threshold. If the operational threshold is 0.2, distance from 0.5 may not represent proximity to the actual decision boundary.


,count,mean_confidence,mean_score
error_type,,,
true_positive,2650,0.223640,0.723640
true_negative,1535,0.177244,0.322756
false_positive,1081,0.155138,0.655138
false_negative,734,0.125064,0.374936


## 13. Hard-case mining

A mature evaluation process does not sample errors uniformly.

Prioritize:

- high-confidence errors,
- high-cost errors,
- recurring error patterns,
- errors near operational boundaries,
- examples representing emerging populations.

This creates a feedback loop from evaluation into data curation.

The notebook builds two tables:
⦁	hard_fn: false negatives, sorted by increasing predicted probability.
⦁	hard_fp: false positives, sorted by decreasing predicted probability.
⦁	It then displays the top 15 from each table.
A false negative with a very low probability is a confident miss. A false positive with a very high probability is a confident false alarm.

These cases can reveal:
⦁	Incorrect or inconsistent labels.
⦁	Missing predictive features.
⦁	Poorly represented populations.
⦁	Unusual but legitimate cases.
⦁	Incorrect business rules or thresholds.
⦁	Feature pipelines that do not match training assumptions.
⦁	Important caveat: The code ranks errors by confidence but does not classify their root causes or estimate their business costs. Hard-case mining should lead to a structured review and data-curation process—not automatic retraining on every error.


In [16]:
# High-confidence false negatives
hard_fn = errors[
    (errors["error_type"] == "false_negative")
].sort_values(
    "prediction_probability", ascending=True
)

# High-confidence false positives
hard_fp = errors[
    (errors["error_type"] == "false_positive")
].sort_values(
    "prediction_probability", ascending=False
)

print("Hard false negatives:")
display(hard_fn[[
    "customer_id", "period", "x1", "x2", "x3", "x4",
    "target", "prediction_probability"
]].head(15))

print("Hard false positives:")
display(hard_fp[[
    "customer_id", "period", "x1", "x2", "x3", "x4",
    "target", "prediction_probability"
]].head(15))

Hard false negatives:


,customer_id,period,x1,x2,x3,x4,target,prediction_probability
22417,5604,1,-1.634089,1.784215,-0.856741,-0.942186,1,0.075563
615,153,3,-1.282635,1.111938,-1.615930,-2.440439,1,0.089851
16904,4226,0,-1.394008,1.289069,-2.125801,NaN,1,0.094897
229,57,1,-0.287383,2.985593,0.140159,-1.991227,1,0.110920
4519,1129,3,-1.738815,1.055197,-0.392762,-0.867086,1,0.122912
20965,5241,1,-1.521865,1.113696,-0.887384,-0.680289,1,0.123358
6080,1520,0,-0.812401,1.731262,-1.105239,-1.061656,1,0.125902
15421,3855,1,-0.993091,1.383725,-1.531835,-0.676305,1,0.126857
5335,1333,3,-1.060154,2.179826,0.364395,-0.643135,1,0.134086
13651,3412,3,-0.183187,2.871961,-1.305720,1.148422,1,0.136027


Hard false positives:


,customer_id,period,x1,x2,x3,x4,target,prediction_probability
13040,3260,0,1.798002,-1.556614,1.361118,1.544584,0,0.967268
4141,1035,1,3.454046,-0.442903,-0.018759,0.418027,0,0.965284
1100,275,0,1.130099,-2.395940,-0.545367,0.818290,0,0.934548
21848,5462,0,0.693622,-1.099989,2.111963,1.534065,0,0.923222
3847,961,3,2.739504,NaN,0.707825,-1.158073,0,0.921163
18864,4716,0,2.218107,0.116929,1.352699,-0.048121,0,0.915292
5871,1467,3,2.447803,0.027098,0.337893,0.483105,0,0.915273
14869,3717,1,0.666439,-1.604448,0.757613,1.644531,0,0.914241
20941,5235,1,2.211909,-1.133057,-1.761504,1.066571,0,0.913153
17398,4349,2,1.359178,-0.943946,0.813386,0.715502,0,0.912359


## 14. Perturbation / robustness testing

Evaluation should test whether small plausible input changes cause unacceptable output changes.

Examples:
- missing a non-critical field,
- rounding numerical inputs,
- delayed feature availability,
- categorical value changes,
- measurement noise,
- stale features.

This is not adversarial ML research; it is **operational robustness testing**.

The function perturbation_test() first calculates baseline probabilities. It then applies each named perturbation to a copy of the input data and calculates probabilities again.
It reports three measures:
⦁	mean_abs_probability_change: average absolute change in probability.
⦁	max_abs_probability_change: largest absolute change.
⦁	flip_rate_at_0_5: fraction of predictions that switch class at threshold 0.5.
The notebook tests three perturbations:
1.	Increase x1 by 5%.
2.	Set x2 to missing.
3.	Round x3 to one decimal place.

This tests sensitivity, not necessarily correctness. A change in probability is not automatically a defect. It is a defect only if the change is unacceptable for the intended business use.
For example, rounding a feature may cause negligible changes, while a missing critical feature might change a decision enough to require fallback behavior or human review.
Technical caveat: The test currently measures changes in scores and decisions, not degradation in predictive accuracy. Since the true labels are available in this synthetic exercise, a stronger evaluation would also calculate metrics under each perturbation and compare them with the baseline.


In [17]:
def perturbation_test(model, X, perturbations):
    baseline = model.predict_proba(X)[:, 1]
    rows = []

    for name, fn in perturbations.items():
        Xp = fn(X.copy())
        pp = model.predict_proba(Xp)[:, 1]

        rows.append({
            "perturbation": name,
            "mean_abs_probability_change": np.mean(np.abs(pp - baseline)),
            "max_abs_probability_change": np.max(np.abs(pp - baseline)),
            "flip_rate_at_0_5": np.mean(
                (baseline >= 0.5) != (pp >= 0.5)
            )
        })

    return pd.DataFrame(rows)

X_probe = X_test.copy()

perturbations = {
    "x1_plus_5_percent": lambda x: x.assign(x1=x["x1"] * 1.05),
    "x2_missing": lambda x: x.assign(x2=np.nan),
    "x3_rounding": lambda x: x.assign(x3=np.round(x["x3"], 1)),
}

perturbation_test(model_a, X_probe, perturbations)

,perturbation,mean_abs_probability_change,max_abs_probability_change,flip_rate_at_0_5
0,x1_plus_5_percent,0.005580,0.024442,0.008667
1,x2_missing,0.098113,0.483716,0.179333
2,x3_rounding,0.001612,0.004010,0.001833


## 15. Out-of-distribution stress test

A model can perform well on IID data and poorly on a shifted population. Here we artificially shift one feature and measure degradation.

In production, this corresponds to:
- new geography,
- new product,
- changed customer mix,
- pricing change,
- sensor upgrade,
- policy change.

A model may perform well on data similar to its training data but poorly when the population or data-generating process changes.
This section deliberately shifts a feature to simulate a changed input distribution.
X_shifted = X_test.copy()X_shifted["x1"] = X_shifted["x1"] + 1.5. The code adds 1.5 to every test-set value of x1, then calculates new probabilities.

It compares baseline and shifted:
⦁	ROC-AUC.
⦁	Brier score.
⦁	Mean predicted score.
⦁	What to explain to the class

A shift in the mean predicted score may indicate that the model is reacting strongly to a population change. A drop in ROC-AUC indicates poorer ranking on the shifted examples, while an increase in Brier score indicates worse probability accuracy.
Important caveat: The shifted data is artificial, and x1 + 1.5 may or may not correspond to a plausible real-world scenario. In a real system, stress scenarios should be defined with domain experts. Also, a score shift alone is not proof of degraded predictive performance; use the known outcomes to assess actual metric degradation.


In [18]:
X_shifted = X_test.copy()
X_shifted["x1"] = X_shifted["x1"] + 1.5

p_shifted = model_a.predict_proba(X_shifted)[:, 1]

stress_results = {
    "baseline_auc": roc_auc_score(y_test, pa),
    "shifted_auc": roc_auc_score(y_test, p_shifted),
    "baseline_brier": brier_score_loss(y_test, pa),
    "shifted_brier": brier_score_loss(y_test, p_shifted),
    "mean_score_shift": p_shifted.mean() - pa.mean()
}

pd.Series(stress_results)

baseline_auc        0.754816
shifted_auc         0.754816
baseline_brier      0.198344
shifted_brier       0.247446
mean_score_shift    0.207295
dtype: float64

## 16. Benchmark harness

A benchmark should make model comparisons repeatable and preserve both model-quality and operational measurements.

Store:
- model name/version,
- dataset version,
- split strategy,
- random seed,
- metric values,
- threshold,
- latency if available,
- training time,
- model size,
- notes.

This makes evaluation an engineering artifact rather than a notebook screenshot.

The models dictionary contains three candidates:
⦁	Logistic regression.
⦁	Random forest.
⦁	Gradient boosting.

The loop fits each model and measures elapsed training time with perf_counter(). It then measures the time required to generate predictions on the test set.

For each model, it calls evaluate_binary_classifier() and records the metrics alongside:
⦁	Training time.
⦁	Total inference time for the test set.
⦁	Average inference time per row.
⦁	The resulting DataFrame is sorted by ROC-AUC.

This is an important transition from evaluating models in isolation to comparing candidates under a consistent evaluation protocol.

A model with the highest AUC is not necessarily the best production choice. The team should also consider calibration, business cost, inference latency, throughput, memory footprint, interpretability, and operational complexity.

Technical caveats:
⦁	The latency measure is a single batch timing, not a robust latency benchmark. It does not report p50/p95/p99 latency, warm-up effects, concurrency, or throughput under load.
⦁	Training time depends on the hardware and software environment.
⦁	The models are evaluated at threshold 0.5, which may not be optimal for every candidate.
⦁	The models are fitted on the same training data and compared on the same test set, which is useful, but final model selection should not repeatedly optimize against that test set.


In [19]:
from time import perf_counter

models = {
    "logistic_regression": model_a,
    "random_forest": model_b,
    "gradient_boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
    ])
}

benchmark_rows = []

for name, model in models.items():
    start = perf_counter()
    model.fit(X_train, y_train)
    train_seconds = perf_counter() - start

    start = perf_counter()
    prob = model.predict_proba(X_test)[:, 1]
    inference_seconds = perf_counter() - start

    metrics = evaluate_binary_classifier(
        y_test, prob, threshold=0.5
    )

    benchmark_rows.append({
        "model": name,
        "train_seconds": train_seconds,
        "inference_seconds_total": inference_seconds,
        "inference_ms_per_row": inference_seconds / len(X_test) * 1000,
        **metrics
    })

benchmark = pd.DataFrame(benchmark_rows)
benchmark.sort_values("roc_auc", ascending=False)

,model,train_seconds,inference_seconds_total,inference_ms_per_row,n,prevalence,roc_auc,pr_auc,log_loss,brier,accuracy,balanced_accuracy,precision,recall,specificity,f1,tn,fp,fn,tp,threshold
0,logistic_regression,0.016879,0.003579,0.000597,6000,0.564,0.754816,0.789697,0.580487,0.198344,0.697500,0.684935,0.710265,0.783097,0.586774,0.744905,1535,1081,734,2650,0.5
2,gradient_boosting,1.867078,0.004526,0.000754,6000,0.564,0.750776,0.787323,0.584239,0.199897,0.692667,0.680867,0.708559,0.773050,0.588685,0.739401,1540,1076,768,2616,0.5
1,random_forest,1.235757,0.105644,0.017607,6000,0.564,0.743445,0.776115,0.592551,0.202838,0.687667,0.675350,0.703394,0.771572,0.579128,0.735908,1515,1101,773,2611,0.5


## 17. Regression evaluation under asymmetric business loss

RMSE/MAE are not always sufficient. Suppose under-predicting demand costs twice as much as over-predicting demand.

Then evaluation should use a weighted business loss.

Standard regression metrics treat errors symmetrically or aggregate them in ways that may not match business costs.
For example, underestimating demand might cause lost sales, while overestimating demand might cause inventory holding costs. The two errors need not be equally expensive.

Two models can have similar RMSE but different business costs if their errors differ in direction. Model evaluation should therefore reflect the decision being made, not just a conventional metric.

Technical caveat: The asymmetric loss is a simplified illustration. Real cost functions may be nonlinear, depend on the size of the error, vary by business segment, or include fixed costs and capacity constraints



In [20]:
# Synthetic regression example
Xr, yr = make_regression(
    n_samples=5000,
    n_features=12,
    noise=15,
    random_state=RANDOM_STATE
)

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.25, random_state=RANDOM_STATE
)

reg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

reg.fit(Xr_train, yr_train)
yr_hat = reg.predict(Xr_test)

def asymmetric_regression_loss(y_true, y_pred, under_cost=2.0, over_cost=1.0):
    error = np.asarray(y_pred) - np.asarray(y_true)
    return np.mean(
        np.where(error < 0, under_cost * np.abs(error), over_cost * np.abs(error))
    )

print("MAE :", mean_absolute_error(yr_test, yr_hat))
print("RMSE:", mean_squared_error(yr_test, yr_hat) ** 0.5)
print("R2  :", r2_score(yr_test, yr_hat))
print("Business loss:", asymmetric_regression_loss(yr_test, yr_hat))

MAE : 11.972573885294569
RMSE: 14.941412857655928
R2  : 0.9943026588894452
Business loss: 17.88773059918082


## 18. Reproducible evaluation report

A production evaluation report should be machine-generated.

The following function creates a compact model card-style evaluation summary.

It creates a common structure that can later be stored in an experiment tracker, model registry, or audit system.

Important caveat: The metadata values are manually supplied labels. The function does not independently verify the dataset version, model artifact, code commit, training configuration, or actual split used. A production implementation should capture these from the pipeline rather than rely on manual declarations.


In [21]:
def evaluation_report(
    model_name,
    dataset_name,
    split_strategy,
    y_true,
    y_prob,
    threshold=0.5,
    metadata=None
):
    metrics = evaluate_binary_classifier(
        y_true, y_prob, threshold=threshold
    )

    report = {
        "model_name": model_name,
        "dataset": dataset_name,
        "split_strategy": split_strategy,
        "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
        "metrics": metrics,
        "metadata": metadata or {}
    }
    return report

report = evaluation_report(
    model_name="logistic_regression_v1",
    dataset_name="customer_eval_v3",
    split_strategy="group_holdout",
    y_true=y_test,
    y_prob=pa,
    threshold=0.5,
    metadata={
        "random_state": RANDOM_STATE,
        "feature_count": len(features),
        "business_context": "customer risk prioritization"
    }
)

report

{'model_name': 'logistic_regression_v1',
 'dataset': 'customer_eval_v3',
 'split_strategy': 'group_holdout',
 'timestamp_utc': '2026-10-24T03:47:02.212772+00:00',
 'metrics': {'n': 6000,
  'prevalence': np.float64(0.564),
  'roc_auc': 0.7548155648816883,
  'pr_auc': 0.7896969609203831,
  'log_loss': 0.5804866084878836,
  'brier': 0.19834414874342082,
  'accuracy': 0.6975,
  'balanced_accuracy': 0.6849353135098792,
  'precision': 0.7102653444116859,
  'recall': 0.783096926713948,
  'specificity': np.float64(0.5867737003058104),
  'f1': 0.7449051300070274,
  'tn': np.int64(1535),
  'fp': np.int64(1081),
  'fn': np.int64(734),
  'tp': np.int64(2650),
  'threshold': 0.5},
 'metadata': {'random_state': 42,
  'feature_count': 4,
  'business_context': 'customer risk prioritization'}}

## 19. Capstone: evaluation checklist

Before approving a model for deployment, the evaluation pipeline should answer:

### Dataset
- Is the evaluation population representative?
- Is the label definition stable?
- Is there temporal leakage?
- Is entity leakage possible?
- Are missingness patterns realistic?

### Metrics
- Which metric represents the business decision?
- Are threshold-free and threshold-dependent metrics separated?
- Are confidence intervals reported?
- Is calibration evaluated where probabilities matter?

### Robustness
- What happens under missing features?
- What happens under distribution shift?
- What happens for rare slices?
- What are the high-confidence errors?

### Decision
- What is the intervention capacity?
- What is the cost of false positives and false negatives?
- Is the threshold selected independently of the final test set?

### Reproducibility
- Dataset version?
- Code version?
- Model version?
- Random seed?
- Split strategy?
- Environment?

**Recommended classroom exercise:** require teams to reject at least one apparently "better" model after discovering a leakage, calibration, subgroup, temporal, or business-cost issue.